<center><img src="../../image/dice.png" alt="DICE" style="width: 300px;"/></center>

# 6.机械臂抓取区域标定

@－－－－DICE－－－－<br>
@  文件名：6.机械臂抓取区域标定.ipynb <br>
@  版本：V2.0 <br>
@  说明：Jetson Nano 机械臂基本运动<br>
本实验属于机械臂基础运动实验部分<br>

## 导入依赖(运行程序前，请先关闭其他程序内核)

In [1]:
import math
import numpy as np
import yaml
# 自定义库 
from transform import Transform
from icp import icp_solver_svd
from arm5dof_uservo import Arm5DoFUServo
from config import *

## 载入九点标定的配置文件

In [2]:
# 载入九点标定法的配置文件
handeye_9points_config = None
with open("config/handeye_9points.yaml", 'r', encoding='utf-8') as f:
    handeye_9points_config = yaml.load(f.read(), Loader=yaml.SafeLoader)

## 创建机械臂

In [3]:
# 创建机械臂
arm = Arm5DoFUServo(is_init_pose=False)

In [4]:
# 设置为阻尼模式
arm.set_damping(0)

## 获取采样点

将机械臂末端拖拽到9个点的位置，记录下机械臂末端的坐标值(x, y, z)

![](./image/p0.jpg)

In [5]:
p0 = arm.get_tool_pose()[:3]
p0

[246.48409306416937, 55.08676367392171, 4.107263448246073]

![](./image/p1.jpg)

In [6]:
p1 = arm.get_tool_pose()[:3]
p1

[243.42214996174678, -1.7127610695147075, -0.6942661992453338]

![](./image/p2.jpg)

In [7]:
p2 = arm.get_tool_pose()[:3]
p2

[238.49329054340404, -66.2330958404492, 0.18930789580483065]

![](./image/p3.jpg)

In [8]:
p3 = arm.get_tool_pose()[:3]
p3

[203.52227204952118, 60.396556161127556, 8.194005403242187]

In [9]:
p4 = arm.get_tool_pose()[:3]
p4

[200.03058669460958, 1.0555801766721378, 2.93566569583146]

![](./image/p5.jpg)

In [10]:
p5 = arm.get_tool_pose()[:3]
p5

[203.26785708580977, -65.41400892213778, 8.176393265465208]

![](./image/p6.jpg)

In [11]:
p6 = arm.get_tool_pose()[:3]
p6

[160.85109037798338, 63.254000894238786, 1.2977189330984515]

![](./image/p7.jpg)

In [12]:
p7 = arm.get_tool_pose()[:3]
p7

[158.99241743104722, 3.3565389961309307, -1.2397464859621152]

![](./image/p8.jpg)

In [13]:
p8 = arm.get_tool_pose()[:3]
p8

[157.4023121756508, -67.0714380166872, 2.1781682450345556]

## 数据保存

In [14]:
ws_9p_at_arm = np.float32([p0, p1, p2, p3, p4, p5, p6, p7, p8])
ws_9p_at_arm

array([[246.484,  55.087,   4.107],
       [243.422,  -1.713,  -0.694],
       [238.493, -66.233,   0.189],
       [203.522,  60.397,   8.194],
       [200.031,   1.056,   2.936],
       [203.268, -65.414,   8.176],
       [160.851,  63.254,   1.298],
       [158.992,   3.357,  -1.24 ],
       [157.402, -67.071,   2.178]], dtype=float32)

将数据存放到csv文件中

In [15]:
np.savetxt('config/arm9points.txt', ws_9p_at_arm, delimiter=',', fmt='%.1f')

## Base标定

In [16]:
# 设置Numpy的打印选项
# 精确位数3，不启用科学计数法
np.set_printoptions(precision=3, suppress=True)

# 载入九点标定法的配置文件
handeye_9points_config = None
with open("config/handeye_9points.yaml", 'r', encoding='utf-8') as f:
	handeye_9points_config = yaml.load(f.read(), Loader=yaml.SafeLoader)


# 载入机械臂末端达到九个点
# 载入九点在2D图像中的坐标
arm_9points = np.loadtxt("config/arm9points.txt", delimiter=',')
# 主要注意的是，对于关节臂构型的机械臂
# 有的在Z轴上的误差很大，因此需要将
# arm_9points里面Z轴坐标替换为Z轴的均值。
z_mean = np.mean(arm_9points[:, 2])

# z轴误差纠正拟合
# - 转化为r跟z的形式
r = np.sqrt(arm_9points[:, 0]**2 + arm_9points[:, 1]**2)
z_error =  arm_9points[:, 2] - z_mean
z_error_polyfit = np.polyfit(r, z_error, 1)
np.savetxt("config/z_error_polyfit.txt", z_error_polyfit, fmt='%.4f', delimiter=",")
arm_9points[:, 2] = z_mean
print(f"9点在机械臂坐标系的坐标: \n{arm_9points}")

# 9点在工作台坐标系的坐标
# ws_9points_df = pd.read_excel("config/九点坐标-工作台坐标系.xlsx")
# ws_9points = np.float64(ws_9points_df.to_numpy())[:, 1:]
w = handeye_9points_config["board_width"]
h = handeye_9points_config["board_height"]
# P0点的坐标
x0 = 0.5 * h
y0 = 0.5 * w
ws_9points = np.float64([
	[x0, y0, 0], 	#P0
	[x0, 0, 0], 	#P1
	[x0, -y0, 0], 	#P2
 	[0, y0, 0], 	#P3
	[0, 0, 0], 		#P4
	[0, -y0, 0], 	#P5
	[-x0, y0, 0], 	#P6
	[-x0, 0, 0], 	#P7
	[-x0, -y0, 0], 	#P8
])
print(f"9点在工作台坐标系的坐标: \n{ws_9points}")

# 求解机械臂基坐标系到工作台坐标系的变换
R, t = icp_solver_svd(arm_9points, ws_9points)
T_arm2ws = np.eye(4)
T_arm2ws[:3, :3] = R
T_arm2ws[:3, 3] = t.reshape(-1)
print("T_arm2ws: \n{}".format(T_arm2ws))
np.savetxt("config/T_arm2ws.txt", T_arm2ws, fmt='%.3f', delimiter=",")

9点在机械臂坐标系的坐标: 
[[246.5  55.1   2.8]
 [243.4  -1.7   2.8]
 [238.5 -66.2   2.8]
 [203.5  60.4   2.8]
 [200.    1.1   2.8]
 [203.3 -65.4   2.8]
 [160.9  63.3   2.8]
 [159.    3.4   2.8]
 [157.4 -67.1   2.8]]
9点在工作台坐标系的坐标: 
[[ 50.   66.5   0. ]
 [ 50.    0.    0. ]
 [ 50.  -66.5   0. ]
 [  0.   66.5   0. ]
 [  0.    0.    0. ]
 [  0.  -66.5   0. ]
 [-50.   66.5   0. ]
 [-50.    0.    0. ]
 [-50.  -66.5   0. ]]
T_arm2ws: 
[[  0.999   0.037   0.    201.389]
 [ -0.037   0.999   0.     -1.9  ]
 [  0.      0.      1.      2.8  ]
 [  0.      0.      0.      1.   ]]
